In [10]:
from transformers import pipeline,AutoTokenizer,AutoModelForSequenceClassification
from bs4 import BeautifulSoup 
import torch 
import re 
import requests
import numpy as np 
import pandas as pd 

In [48]:
#tkz = AutoTokenizer.from_pretrained('frameai/PersianSentiment')
#model =AutoModelForSequenceClassification.from_pretrained('frameai/PersianSentiment')

In [56]:
tkz =AutoTokenizer.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')
model =AutoModelForSequenceClassification.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

In [67]:
r =requests.get('https://www.yelp.com/biz/social-brew-cafe-pyrmont')
soup = BeautifulSoup(r.text, 'html.parser')
regex = re.compile('.*comment.*')
results = soup.find_all('p', {'class':regex})
reviews = [result.text for result in results]

In [68]:
reviews

["Great food and ambiance. Staff very friendly. Delicious. Can't wait to go back and try something else on the menu.",
 "Very cute coffee shop and restaurant. They have a lovely outdoor seating area and several tables inside.  It was fairly busy on a Tuesday morning but we were to grab the last open table. The server was so enjoyable, she chatted and joked with us and provided fast service with our ordering, drinks and meals. The food was very good. We ordered a wide variety and every meal was good to delicious. The sweet potato fries on the Chicken Burger plate were absolutely delicious, some of the best I've ever had. I definitely enjoyed this cafe, the outdoor seating, the service and the food!!",
 "Amazing ambience. Coffee was the best we've had in Sydney. The service and food also great. As must stop for breakfast.",
 "Six of us met here for breakfast before our walk to Manly. We were enjoying visiting with each other so much that I apologize for not taking any photos. We all enjo

In [69]:
df = pd.DataFrame(np.asarray(reviews),columns=['reviews'])

In [70]:
df

,reviews
0,Great food and ambiance. Staff very friendly. ...
1,Very cute coffee shop and restaurant. They hav...
2,Amazing ambience. Coffee was the best we've ha...
3,Six of us met here for breakfast before our wa...
4,Ricotta hot cakes! These were so yummy. I ate ...
5,We came for brunch twice in our week-long visi...
6,Great food amazing coffee and tea. Short walk ...
7,We came for brunch and they ran out of seven s...
8,Great place with delicious food and friendly s...
9,The food was delicious. The ricotta pancakes w...


In [71]:
def ss(reviews):
    token = tkz.encode(reviews,return_tensors='pt')
    res = model(token)
    return int(torch.argmax(res.logits))+1

In [72]:
ss(df['reviews'].iloc[1])

5

In [79]:
for i in range(len(df['reviews'])):
    df['sentiment'].iloc[i] = ss(df['reviews'].iloc[i])
df

/tmp/ipykernel_357853/639416692.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['sentiment'].iloc[i] = ss(df['reviews'].iloc[i])
/tmp/ipykernel_357853/639416692.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a

,reviews,sentiment
0,Great food and ambiance. Staff very friendly. ...,5
1,Very cute coffee shop and restaurant. They hav...,5
2,Amazing ambience. Coffee was the best we've ha...,5
3,Six of us met here for breakfast before our wa...,4
4,Ricotta hot cakes! These were so yummy. I ate ...,5
5,We came for brunch twice in our week-long visi...,5
6,Great food amazing coffee and tea. Short walk ...,5
7,We came for brunch and they ran out of seven s...,2
8,Great place with delicious food and friendly s...,5
9,The food was delicious. The ricotta pancakes w...,4
